In [5]:
#Q.11 카테고리 확대·유지·축소 매트릭스
# 순   카테고리에 걸칠 수 있어서)
# -> 세 지표를 rank()로 방향 통일해 가중 종합점수 산출, 확대/유지/축소 부여
# -> 매출 단독 판단과 달라진 카테고리 지목 + 가중치 민감도 확인

In [6]:
import pandas as pd

orders = pd.read_csv("../data/orders.csv", usecols=["order_id", "status"], dtype={"status": "category"})
items = pd.read_csv("../data/order_items.csv")
products = pd.read_csv("../data/products.csv", usecols=["product_id", "category", "cost"])
products["category"] = products["category"].str.strip()  # " 식품" 등 앞뒤 공백 오염 정리

#데이터 merge - items 을 중심으로
# items를 기준으로 모든 행을 유지하면서, 매칭되는 컬럼을 옆에 추가
df = items.merge(orders)  #items 과 orders에 공통으로 존재하는 컬럼을 기준으로
df = items.merge(orders, on="order_id", how="left").merge(products, on="product_id", how="left")


before = len(df)
df = df.dropna(subset=["unit_price"])
df = df[df["quantity"] > 0]  # unit_price 결측(3.0%), quantity<=0(0/음수, 데이터 오류) 라인 제외
print(f"제외된 라인: {before - len(df):,} / 전체 {before:,}")

#데이터 로드 / 정제 => valid (취소 제외 유효 라인) 생성
valid = df[df["status"] != "canceled"].copy()  # 취소 주문은 매출·마진·반품률 모두에서 제외
valid["revenue"] = valid["quantity"] * valid["unit_price"] * (1 - valid["discount"])
valid["cost_amt"] = valid["quantity"] * valid["cost"]
print(f"유효(취소 제외) 주문 라인: {len(valid):,} / {len(df):,}")

제외된 라인: 15,674 / 전체 502,351
유효(취소 제외) 주문 라인: 437,464 / 486,677


In [7]:
# items.info() #items는 원본
df.info()  #merge 후 만들어진 df

<class 'pandas.DataFrame'>
Index: 486677 entries, 0 to 502350
Data columns (total 9 columns):
 #   Column         Non-Null Count   Dtype   
---  ------         --------------   -----   
 0   order_item_id  486677 non-null  int64   
 1   order_id       486677 non-null  int64   
 2   product_id     486677 non-null  int64   
 3   quantity       486677 non-null  int64   
 4   unit_price     486677 non-null  float64 
 5   discount       486677 non-null  float64 
 6   status         486677 non-null  category
 7   category       486677 non-null  str     
 8   cost           486677 non-null  float64 
dtypes: category(1), float64(3), int64(4), str(1)
memory usage: 36.7 MB


In [8]:
# 1. 카테고리별 순매출·마진율·반품률 결합 (net_revenue, margin_rate, return_rate)
#valid(취소 제외 주문 라인)을 category로 묶어서 category revenue 합 


#category별 column을 가진 dataframe => rev_margin

#rev_Margin이라는 DataFrame 이 만들어지는데 category를 index로 하고, net_revenue(카테고리별 매출 합/순매출)랑 total_cost(카테고리별 원가합)
rev_margin = valid.groupby("category").agg(net_revenue=("revenue", "sum"), total_cost=("cost_amt", "sum"))
#세번째 column인 margin_rate(마진률)   마진률 = (순매출 = 원가) / 순매출
rev_margin["margin_rate"] = (rev_margin["net_revenue"] - rev_margin["total_cost"]) / rev_margin["net_revenue"]


#Boolean column으로 반품 여부를 표시 -> groupby.agg로 "전체개수"와 "반품개수"를 한번에 aggregate
#valid의 각 row마다 status가 "returned" 인지를 비교해서 True/False boolean column is_returned를 만든다
valid["is_returned"] = valid["status"] == "returned"
#category별로 묶어서 2가지를 동시에 집계
ret = valid.groupby("category").agg(
    valid_lines=("status", "size"),
    returned=("is_returned", "sum"),
)
ret["return_rate"] = ret["returned"] / ret["valid_lines"]

#  cat_tbl — 세 지표 결합 (rev_margin에서 필요한 두 column만 골라서 return_rate)
#cat_tbl 이라는 새로운 dataframe을 뽑아냄 -> [] 안에 원하는 콜럼 3개 지정
cat_tbl = rev_margin[["net_revenue", "margin_rate"]].join(ret["return_rate"])
cat_tbl.sort_values("net_revenue", ascending=False).round(4)

,net_revenue,margin_rate,return_rate
category,,,
전자,2.130452e+10,0.2268,0.0554
가구,1.515239e+10,0.2156,0.0565
의류,4.294114e+09,0.2303,0.0566
도서,4.186966e+09,0.2940,0.0537
뷰티,3.269384e+09,0.2207,0.0569
식품,1.760147e+09,0.2322,0.0538


In [ ]:
# 2. 지표별 rank()로 방향 통일(매출·마진은 클수록, 반품률은 작을수록 좋음) 후 가중 종합점수
#rank() 의 기본 동작: ascending=True(기본값)일때는 값이 클수록 더 큰 순위 숫자
#그렇게 때문에 반품율은 ascending=False 로 뒤집어줘야지 더작은 순위는 큰 순위로 바뀜

#score_table에서 매출 큰/마진율 큰/반품율 낮은 -> 큰 ranking 나올 수 있게
#이렇게 해서 세 column 모두 "rank 숫자가 클수록 좋다" 라는 동일한 방향으로 통일됨
def score_table(tbl, w_rev, w_margin, w_return):
    r_rev = tbl["net_revenue"].rank()
    r_margin = tbl["margin_rate"].rank()
    r_return = tbl["return_rate"].rank(ascending=False)  #마진율은 ascending=False

    #가중치 계산할때 가중치 비중 (고정) X rank() 에서 만들어진 점수 
    score = w_rev * r_rev + w_margin * r_margin + w_return * r_return
    n = len(tbl)
    rk = score.rank(ascending=False, method="first")

    #result["decision"]은 (종합)net_revenue + margint_rate + return_rate 3개를 가중합
    decision = pd.cut(rk, bins=[0, n / 3, 2 * n / 3, n], labels=["확대", "유지", "축소"])
    return tbl.assign(score=score, decision=decision).sort_values("score", ascending=False)

# 3. 종합 점수로 확대/유지/축소 부여 (재무 리스크인 마진·반품에 매출보다 약간 더 무게)
W_REV, W_MARGIN, W_RETURN = 0.3, 0.4, 0.3
result = score_table(cat_tbl, W_REV, W_MARGIN, W_RETURN)
result.round(4)

,net_revenue,margin_rate,return_rate,score,decision
category,,,,,
도서,4.186966e+09,0.2940,0.0537,5.1,확대
전자,2.130452e+10,0.2268,0.0554,4.2,확대
식품,1.760147e+09,0.2322,0.0538,3.8,유지
의류,4.294114e+09,0.2303,0.0566,3.4,유지
가구,1.515239e+10,0.2156,0.0565,2.8,축소
뷰티,3.269384e+09,0.2207,0.0569,1.7,축소


In [15]:
result["decision"].info()

<class 'pandas.Series'>
Index: 6 entries, 도서 to 뷰티
Series name: decision
Non-Null Count  Dtype   
--------------  -----   
6 non-null      category
dtypes: category(1)
memory usage: 413.0 bytes


In [ ]:
# 매출 단독 판단과 비교 -> 결론이 달라진 카테고리 지목
# 위의 세 지표를 rank() 방향 통일해 종합점수 산출하고 거기에다가 확대/유지/축소 부여
n = len(cat_tbl) #cat_tbl의 row 개수(category 6개)를 저장
#cat_tbl의 net_revenue column만 갖고 순위를 매김
rev_rk = cat_tbl["net_revenue"].rank(ascending=False, method="first")
#rev_rk 를 3구간(확대, 유지, 축소)로 잘라 라벨을 붙힘 

#rev_only는 net_revenue 하나만 (즉 매출만 보면 몇 ranking인가로 라벨링)
#(bins 개수 = labels 개수 + 1).역
rev_only = pd.cut(rev_rk, bins=[0, n / 3, 2 * n / 3, n], labels=["확대", "유지", "축소"])

#매출 단독 판단과 달라진 카테고리 지목 + 가중치 민감도
compare = pd.DataFrame({"매출단독": rev_only, "종합점수": result["decision"]})
print("판단이 달라진 카테고리:")
compare[compare["매출단독"] != compare["종합점수"]] 

판단이 달라진 카테고리:


,매출단독,종합점수
category,,
가구,확대,축소
도서,유지,확대
식품,축소,유지


In [12]:
compare.info()

<class 'pandas.DataFrame'>
Index: 6 entries, 가구 to 전자
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype   
---  ------  --------------  -----   
 0   매출단독    6 non-null      category
 1   종합점수    6 non-null      category
dtypes: category(2)
memory usage: 569.0 bytes


In [11]:
# 4. 가중치 민감도: 매출/마진 비중을 바꿔도 결론이 유지되는지 확인
weight_sets = {
    "기본(.3/.4/.3)": (0.3, 0.4, 0.3),
    "매출중시(.6/.2/.2)": (0.6, 0.2, 0.2),
    "마진중시(.2/.6/.2)": (0.2, 0.6, 0.2),
}
sensitivity = pd.DataFrame({name: score_table(cat_tbl, *w)["decision"] for name, w in weight_sets.items()})
sensitivity

,기본(.3/.4/.3),매출중시(.6/.2/.2),마진중시(.2/.6/.2)
category,,,
가구,축소,유지,축소
도서,확대,확대,확대
뷰티,축소,축소,축소
식품,유지,축소,확대
의류,유지,유지,유지
전자,확대,확대,유지


In [14]:
sensitivity.info()

<class 'pandas.DataFrame'>
Index: 6 entries, 가구 to 전자
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   기본(.3/.4/.3)    6 non-null      category
 1   매출중시(.6/.2/.2)  6 non-null      category
 2   마진중시(.2/.6/.2)  6 non-null      category
dtypes: category(3)
memory usage: 553.0 bytes


### 제출물

- **종합표**: `result` (순매출·마진율·반품률·종합점수·결론, 위 셀)
- **제안**: 확대 = 전자·도서 / 유지 = 식품·의류 / 축소 = 가구·뷰티
- **판단이 바뀐 사례**: 가구는 매출 2위지만 마진율이 6개 카테고리 중 최저(21.6%)이고 반품률도 상위권이라 축소로 내려감. 도서는 매출 최하위권이지만 마진율 최고(29.4%)·반품률 최저라 확대로 올라감. 식품은 매출 꼴찌지만 반품률이 가장 낮아 유지로 올라감.
- **민감도**: 가중치를 매출중시/마진중시로 바꿔도 가구는 한 번도 확대로 오르지 않고 도서는 항상 확대 → 두 결론은 가중치에 안정적. 전자·식품은 가중치에 따라 등급이 갈려 합의가 더 필요한 경계 카테고리.